# 03 - Red neuronal (MLP) en PyTorch

**Objetivo:** construir desde cero un perceptrón multicapa (MLP) en PyTorch que prediga la dirección del día siguiente, con **el mismo corte por fecha, las mismas características y las mismas métricas** del notebook 02, para compararlo de forma justa contra el baseline y la regresión logística.

El documento del proyecto (sección 4.7) pide esto; la tabla dice en qué parte del notebook se cumple cada punto:

| Requisito del documento | Dónde |
|---|---|
| Clase que herede de `nn.Module` con `forward()` explícito | Sección 4, clase `MLP` |
| `Dataset` propio con `__len__` y `__getitem__` | Sección 4, clase `DatasetAcciones` |
| Ciclo completo: forward, pérdida, backward, `optimizer.step()`, `zero_grad()` | Sección 5, función `entrenar_mlp` |
| Separación entrenamiento/validación y métricas por época | Sección 5 (las últimas fechas del entrenamiento se usan para validar) |
| *Early stopping* y guardar el mejor modelo | Sección 5 (se restaura el estado de la mejor época) |
| Serializar con `torch.save()` y cargar sin reentrenar | Sección 10 |

**Lo que NO se espera:** que la red le gane a la regresión logística. En el notebook 02 ningún modelo se distinguió del azar, y la literatura dice que una red no suele mejorar en datos tabulares con tan poca señal. Lo que se evalúa es que esté bien construida, bien validada y comparada con honestidad.

> Aviso: este notebook se escribió sin poder ejecutar PyTorch en el entorno donde se redactó. Las celdas 4 y 5 son pruebas rápidas para detectar cualquier problema de instalación o de código antes del entrenamiento largo.

## 1. Parámetros desde `configs/config.yaml`

Los hiperparámetros de la red (`capas_ocultas`, `dropout`, `learning_rate`, `batch_size`, `epocas_max`, `paciencia`, `weight_decay`) salen del YAML, igual que en el resto del proyecto.

In [ ]:
import copy
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
import yfinance as yf
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
with open(RAIZ / "configs" / "config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

OBJETIVO = cfg["datos"]["tickers_objetivo"]
INICIO = cfg["datos"]["fecha_inicio"]
H = cfg["datos"]["horizonte_dias"]
VENT_RET = cfg["features"]["ventanas_retorno"]
VENT_ROLL = cfg["features"]["ventanas_rolling"]
RSI_N = cfg["features"]["rsi_periodo"]
N_SPLITS = cfg["validacion"]["n_splits"]
GAP = cfg["validacion"]["gap_dias"]
MESES_HOLDOUT = cfg["validacion"]["meses_holdout"]
SEMILLA = cfg["validacion"]["semilla"]
P = cfg["modelos"]["pytorch"]

print("Objetivo:", OBJETIVO)
print("Red:", P)

## 2. Datos, características y corte por fecha

Es **el mismo código del notebook 02** (sin las características de contexto, porque ahí no ayudaron). Se repite aquí para que el notebook sea independiente; al construir `src/` se escribirá una sola vez.

In [ ]:
def cargar_precios(tickers):
    carpeta = RAIZ / "data" / "exploracion"
    f_c, f_v = carpeta / "close.csv", carpeta / "volume.csv"
    if f_c.exists() and f_v.exists():
        c = pd.read_csv(f_c, index_col=0, parse_dates=True)
        v = pd.read_csv(f_v, index_col=0, parse_dates=True)
        if set(tickers) <= set(c.columns):
            return c[tickers], v[tickers]
    datos = yf.download(tickers, start=INICIO, auto_adjust=True, progress=False)
    carpeta.mkdir(parents=True, exist_ok=True)
    datos["Close"].to_csv(f_c)
    datos["Volume"].to_csv(f_v)
    return datos["Close"][tickers], datos["Volume"][tickers]


def rsi(cierre_t, n):
    delta = cierre_t.diff()
    kw = {"alpha": 1 / n, "adjust": False, "min_periods": n}
    ganancia = delta.clip(lower=0).ewm(**kw).mean()
    perdida = (-delta.clip(upper=0)).ewm(**kw).mean()
    return 100 - 100 / (1 + ganancia / perdida)


def features_propias(c, v):
    ret = c.pct_change(fill_method=None)
    f = pd.DataFrame(index=c.index)
    for k in VENT_RET:
        f[f"ret_{k}d"] = c.pct_change(k, fill_method=None)
    for k in VENT_ROLL:
        f[f"media_{k}d"] = ret.rolling(k).mean()
        f[f"desv_{k}d"] = ret.rolling(k).std()
    f[f"rsi_{RSI_N}"] = rsi(c, RSI_N)
    f[f"dist_media_{VENT_ROLL[-1]}d"] = c / c.rolling(VENT_ROLL[-1]).mean() - 1
    f["vol_rel"] = v / v.rolling(VENT_ROLL[-1]).mean()
    return f


cierre, volumen = cargar_precios(OBJETIVO)

paneles = []
for t in OBJETIVO:
    c = cierre[t].dropna()
    v = volumen[t].reindex(c.index)
    f = features_propias(c, v)
    futuro = c.shift(-H)
    f["y"] = (futuro > c).astype(int)
    f = f[futuro.notna()]
    f["ticker"] = t
    paneles.append(f.rename_axis("fecha"))

panel = pd.concat(paneles).reset_index()
panel = panel.replace([np.inf, -np.inf], np.nan).dropna()
panel = panel.sort_values(["fecha", "ticker"]).reset_index(drop=True)

FECHA_CORTE = pd.Timestamp(cfg["validacion"]["fecha_corte"])
adelante = panel[panel["fecha"] > FECHA_CORTE].reset_index(drop=True)
panel = panel[panel["fecha"] <= FECHA_CORTE].reset_index(drop=True)
print(
    "Fecha de corte:",
    FECHA_CORTE.date(),
    f"| filas posteriores (prueba hacia adelante): {len(adelante):,}",
)

COLS = [c for c in panel.columns if c not in ["fecha", "ticker", "y"]]
print("Filas:", len(panel), "| características:", len(COLS))
print(COLS)

In [ ]:
fechas = np.sort(panel["fecha"].unique())
corte = pd.Timestamp(fechas[-1]) - pd.DateOffset(months=MESES_HOLDOUT)
i_hold = int(np.searchsorted(fechas, np.datetime64(corte)))
fin_train = fechas[i_hold - GAP - 1]

train = panel[panel["fecha"] <= fin_train].reset_index(drop=True)
hold = panel[panel["fecha"] >= fechas[i_hold]].reset_index(drop=True)

X_all = train[COLS].to_numpy()
y_all = train["y"].to_numpy()
fechas_all = train["fecha"].to_numpy()

print(
    "Train  :",
    train["fecha"].min().date(),
    "->",
    train["fecha"].max().date(),
    f"({len(train):,} filas)",
)
print(
    "Holdout:",
    hold["fecha"].min().date(),
    "->",
    hold["fecha"].max().date(),
    f"({len(hold):,} filas)",
)
print(f"Días al alza en entrenamiento: {y_all.mean():.1%}")

**Validación cruzada temporal** (la misma del notebook 02, para que los números sean comparables) y una función para separar, **dentro de cada entrenamiento**, las últimas fechas que servirán de validación para el *early stopping*. Esa separación también es por fechas y con hueco (`gap`): nunca se valida con datos anteriores a los del entrenamiento.

In [ ]:
def splits_por_fecha(fechas_fila, n_splits, gap):
    unicas = np.sort(pd.unique(fechas_fila))
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    salida = []
    for i_tr, i_va in tscv.split(unicas):
        tr = np.flatnonzero(np.isin(fechas_fila, unicas[i_tr]))
        va = np.flatnonzero(np.isin(fechas_fila, unicas[i_va]))
        salida.append((tr, va))
    return salida


def dividir_por_fecha(fechas_fila, frac_val, gap):
    unicas = np.sort(pd.unique(fechas_fila))
    n_val = int(np.ceil(len(unicas) * frac_val))
    f_val = unicas[-n_val:]
    f_tr = unicas[: len(unicas) - n_val - gap]
    idx_tr = np.flatnonzero(np.isin(fechas_fila, f_tr))
    idx_va = np.flatnonzero(np.isin(fechas_fila, f_val))
    return idx_tr, idx_va


CV = splits_por_fecha(fechas_all, N_SPLITS, GAP)
FRAC_ES = P["fraccion_validacion"]
SCORING = {"f1": "f1", "auc": "roc_auc"}

idx_tr, idx_es = dividir_por_fecha(fechas_all, FRAC_ES, GAP)


def rango(idx):
    f = pd.to_datetime(fechas_all[idx])
    return f.min().date(), "->", f.max().date()


print("Entrenamiento interno:", *rango(idx_tr))
print("Validación (early stopping):", *rango(idx_es))
print(f"Filas: {len(idx_tr):,} entrenamiento | {len(idx_es):,} validación")

## 3. Referencias de scikit-learn con los mismos pliegues

Antes de la red, se calculan el baseline y la regresión logística **con los mismos 5 pliegues**. Se usa la logística con los mejores hiperparámetros que encontró el notebook 02 (`C=0.1`, penalización l1, escalador estándar).

In [ ]:
referencias = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistica": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(solver="liblinear", C=0.1, l1_ratio=1)),
        ]
    ),
}

filas_ref = []
for nombre, est in referencias.items():
    r = cross_validate(est, train[COLS], train["y"], cv=CV, scoring=SCORING)
    filas_ref.append(
        {
            "modelo": nombre,
            "auc_cv": r["test_auc"].mean(),
            "auc_std": r["test_auc"].std(),
            "f1_cv": r["test_f1"].mean(),
        }
    )
pd.DataFrame(filas_ref).round(4)

## 4. La red: `Dataset` y `nn.Module`

**Conceptos básicos (para explicarlos en la exposición):**

- Un **tensor** es un arreglo como los de NumPy, pero con dos extras: puede vivir en GPU y recuerda las operaciones que se le hicieron para poder calcular derivadas (*autograd*).
- `Dataset` le dice a PyTorch cómo obtener **una** muestra (`__getitem__`) y cuántas hay (`__len__`). `DataLoader` arma lotes (*batches*) y los mezcla en cada época.
- `nn.Module` es la base de toda red: en `__init__` se declaran las capas y `forward` dice cómo viaja un lote de entrada hasta la salida.
- La red devuelve **logits** (números reales sin límite), no probabilidades. La pérdida `BCEWithLogitsLoss` aplica el *sigmoide* por dentro de forma numéricamente estable. Para obtener probabilidades en inferencia se aplica `torch.sigmoid` a mano.
- **ReLU** pone en cero lo negativo y deja pasar lo positivo: da no linealidad. **Dropout** apaga al azar una fracción de neuronas en cada paso de entrenamiento para que la red no memorice; se desactiva con `modelo.eval()`.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cpu")  # el contenedor de Docker usará solo CPU
torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)
print("PyTorch", torch.__version__, "| dispositivo:", DEVICE)


class DatasetAcciones(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class MLP(nn.Module):
    def __init__(self, n_entrada, capas_ocultas, dropout):
        super().__init__()
        capas = []
        previo = n_entrada
        for n in capas_ocultas:
            capas += [nn.Linear(previo, n), nn.ReLU(), nn.Dropout(dropout)]
            previo = n
        capas.append(nn.Linear(previo, 1))
        self.red = nn.Sequential(*capas)

    def forward(self, x):
        return self.red(x).squeeze(-1)


modelo_demo = MLP(len(COLS), P["capas_ocultas"], P["dropout"])
print(modelo_demo)
n_param = sum(p.numel() for p in modelo_demo.parameters())
print("Parámetros entrenables:", n_param)

### Pruebas rápidas antes de entrenar

Dos pruebas clásicas para descartar errores de código antes de gastar tiempo:

1. **Pérdida inicial.** Con pesos aleatorios la red no sabe nada y predice cerca de 0.5, así que la pérdida debe ser cercana a `ln 2 = 0.693`.
2. **Memorizar un lote chico.** Si la red no logra bajar mucho la pérdida con solo 64 filas y muchas repeticiones, hay un error en el ciclo de entrenamiento (con tan pocos datos tiene que poder memorizarlos).

In [ ]:
Xs = StandardScaler().fit_transform(X_all[:2000])
xb = torch.as_tensor(Xs[:64], dtype=torch.float32)
yb = torch.as_tensor(y_all[:64], dtype=torch.float32)

probar = MLP(len(COLS), [32, 16], 0.0)
fn = nn.BCEWithLogitsLoss()
print("Forma de la salida:", tuple(probar(xb).shape), "(esperado: (64,))")
print(f"Pérdida inicial: {fn(probar(xb), yb).item():.3f} (esperado: cerca de 0.693)")

opt = torch.optim.Adam(probar.parameters(), lr=0.01)
for _ in range(300):
    perdida = fn(probar(xb), yb)
    perdida.backward()
    opt.step()
    opt.zero_grad()
print(f"Pérdida tras memorizar 64 filas: {perdida.item():.3f} (esperado: < 0.2)")

## 5. Ciclo de entrenamiento con *early stopping*

Cada **época** recorre todo el entrenamiento en lotes. Por cada lote: **forward** (predecir), **pérdida** (qué tan mal), **backward** (derivadas de la pérdida respecto a cada peso), **`optimizer.step()`** (mover los pesos) y **`zero_grad()`** (borrar las derivadas acumuladas; si no, se suman entre lotes).

Al terminar cada época se mide la pérdida y el AUC en la **validación** (las últimas fechas del entrenamiento). Si el **AUC de validación** no mejora durante `paciencia` épocas seguidas, se detiene el entrenamiento y se **restaura el estado de la mejor época** (*checkpoint*). Se para por AUC y no por pérdida porque al final lo que importa es el **orden** de las probabilidades; la pérdida premia estar bien calibrado y puede dar la vuelta en otra época (la pérdida se registra igual para las curvas). Así la red no sigue aprendiendo ruido del entrenamiento.

El escalador se ajusta **solo con el entrenamiento** de cada ajuste y se aplica a la validación; si se ajustara con todo habría fuga de datos.

In [ ]:
@torch.no_grad()
def predecir_proba(modelo, X):
    modelo.eval()
    x = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    return torch.sigmoid(modelo(x)).cpu().numpy()


@torch.no_grad()
def evaluar(modelo, X, y, fn_perdida):
    modelo.eval()
    x = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    logits = modelo(x)
    perdida = fn_perdida(logits, torch.as_tensor(y, dtype=torch.float32)).item()
    probs = torch.sigmoid(logits).cpu().numpy()
    return perdida, roc_auc_score(y, probs)


def entrenar_mlp(X_tr, y_tr, X_va, y_va, params, semilla=SEMILLA, verbose=False):
    escalador = StandardScaler().fit(X_tr)
    X_tr_s, X_va_s = escalador.transform(X_tr), escalador.transform(X_va)

    torch.manual_seed(semilla)
    modelo = MLP(X_tr.shape[1], params["capas_ocultas"], params["dropout"])
    modelo.to(DEVICE)
    fn_perdida = nn.BCEWithLogitsLoss()
    optimizador = torch.optim.Adam(
        modelo.parameters(),
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"],
    )
    generador = torch.Generator().manual_seed(semilla)
    cargador = DataLoader(
        DatasetAcciones(X_tr_s, y_tr),
        batch_size=params["batch_size"],
        shuffle=True,
        generator=generador,
    )

    historia = []
    mejor_auc, mejor_epoca, mejor_estado, sin_mejora = -np.inf, 0, None, 0
    for epoca in range(1, params["epocas_max"] + 1):
        modelo.train()
        suma, n = 0.0, 0
        for xb, yb in cargador:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = modelo(xb)  # forward
            perdida = fn_perdida(logits, yb)  # pérdida
            perdida.backward()  # backward
            optimizador.step()  # actualiza pesos
            optimizador.zero_grad()  # borra derivadas
            suma += perdida.item() * len(yb)
            n += len(yb)

        perdida_va, auc_va = evaluar(modelo, X_va_s, y_va, fn_perdida)
        historia.append(
            {
                "epoca": epoca,
                "perdida_train": suma / n,
                "perdida_val": perdida_va,
                "auc_val": auc_va,
            }
        )
        if auc_va > mejor_auc + 1e-5:  # epsilon: el ruido no reinicia la espera
            mejor_auc, mejor_epoca, sin_mejora = auc_va, epoca, 0
            mejor_estado = {
                "epoca": epoca,
                "mejor_auc": float(auc_va),
                "modelo": copy.deepcopy(modelo.state_dict()),
                "optimizador": copy.deepcopy(optimizador.state_dict()),
            }
        else:
            sin_mejora += 1
        if verbose:
            marca = "*" if sin_mejora == 0 else ""
            print(
                f"época {epoca:2d} | train {suma / n:.4f} | val {perdida_va:.4f} "
                f"| auc val {auc_va:.4f} {marca}"
            )
        if sin_mejora >= params["paciencia"]:
            break

    modelo.load_state_dict(mejor_estado["modelo"])  # restaura la mejor época
    return {
        "modelo": modelo,
        "escalador": escalador,
        "epoca_mejor": mejor_epoca,
        "checkpoint": mejor_estado,
        "historia": pd.DataFrame(historia),
    }


def proba(resultado, X):
    return predecir_proba(resultado["modelo"], resultado["escalador"].transform(X))


print("Funciones definidas")

### Primer entrenamiento con los hiperparámetros del YAML

Una sola corrida, con el registro por época visible. Una marca `*` indica una época que mejoró el AUC de validación (esa es la que se guarda).

In [ ]:
PARAMS = {
    "capas_ocultas": P["capas_ocultas"],
    "dropout": P["dropout"],
    "learning_rate": P["learning_rate"],
    "weight_decay": P["weight_decay"],
    "batch_size": P["batch_size"],
    "epocas_max": P["epocas_max"],
    "paciencia": P["paciencia"],
}

t0 = time.perf_counter()
res = entrenar_mlp(
    X_all[idx_tr], y_all[idx_tr], X_all[idx_es], y_all[idx_es], PARAMS, verbose=True
)
print(f"Mejor época: {res['epoca_mejor']} | tiempo: {time.perf_counter() - t0:.0f} s")

In [ ]:
h = res["historia"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(h["epoca"], h["perdida_train"], marker="o", label="Entrenamiento")
ax1.plot(h["epoca"], h["perdida_val"], marker="o", label="Validación")
ax1.axvline(res["epoca_mejor"], color="green", linestyle="--", label="Mejor época")
ax1.axhline(np.log(2), color="k", linestyle=":", label="Sin información (ln 2)")
ax1.set_title("Pérdida por época")
ax1.set_xlabel("Época")
ax1.set_ylabel("Pérdida (entropía cruzada binaria)")
ax1.legend()

ax2.plot(
    h["epoca"], h["auc_val"], marker="o", color="tab:orange", label="AUC validación"
)
ax2.axhline(0.5, color="k", linestyle=":", label="Sin información (0.50)")
ax2.axvline(res["epoca_mejor"], color="green", linestyle="--", label="Mejor época")
ax2.set_title("AUC en validación por época")
ax2.set_xlabel("Época")
ax2.set_ylabel("AUC-ROC")
ax2.legend()
plt.show()

**Cómo leer las curvas.** La pérdida de entrenamiento suele bajar siempre (la red memoriza); la de validación baja un poco y luego sube. El entrenamiento se detiene según el AUC de validación (la línea verde marca la época restaurada). Si la de validación nunca baja de `ln 2 = 0.693`, la red no aprendió nada que sirva fuera del entrenamiento, que es lo esperable con esta señal.

## 6. Exploración de hiperparámetros de la red

Como no hay `GridSearchCV` para PyTorch, la búsqueda es un ciclo manual sobre una rejilla chica: arquitectura (`capas_ocultas`), `dropout` y `weight_decay` (regularización L2). Se evalúa cada combinación con el mismo corte entrenamiento/validación de arriba y se elige por AUC de validación.

Dos advertencias: (1) la validación que elige el mejor es la misma que detiene el entrenamiento, así que el AUC elegido queda un poco optimista; el AUC honesto es el de la validación cruzada de la sección 8. (2) Con señal tan débil, las diferencias entre combinaciones son ruido; la rejilla sirve sobre todo para **documentar qué se probó**.

In [ ]:
rejilla = [
    {"capas_ocultas": capas, "dropout": d, "weight_decay": wd}
    for capas in ([16], [32, 16], [64, 32])
    for d in (0.2, 0.5)
    for wd in (0.0, 1e-3)
]

filas = []
for combo in rejilla:
    t0 = time.perf_counter()
    r = entrenar_mlp(
        X_all[idx_tr], y_all[idx_tr], X_all[idx_es], y_all[idx_es], {**PARAMS, **combo}
    )
    mejor = r["historia"].loc[r["historia"]["epoca"] == r["epoca_mejor"]].iloc[0]
    filas.append(
        {
            **combo,
            "epoca_mejor": r["epoca_mejor"],
            "perdida_val": mejor["perdida_val"],
            "auc_val": mejor["auc_val"],
            "segundos": time.perf_counter() - t0,
        }
    )
    print(f"{combo} -> auc val {mejor['auc_val']:.4f}")

exploracion = pd.DataFrame(filas).sort_values("auc_val", ascending=False)
exploracion.reset_index(drop=True).round(4)

In [ ]:
mejor_combo = exploracion.iloc[0]
PARAMS_FINAL = {
    **PARAMS,
    "capas_ocultas": list(mejor_combo["capas_ocultas"]),
    "dropout": float(mejor_combo["dropout"]),
    "weight_decay": float(mejor_combo["weight_decay"]),
}
print(
    "Combinación elegida:",
    {k: PARAMS_FINAL[k] for k in ("capas_ocultas", "dropout", "weight_decay")},
)

## 7. Variabilidad por la inicialización

Una red empieza con pesos aleatorios, y eso cambia el resultado. Se repite el entrenamiento con 3 semillas distintas y se mide cuánto varía el AUC de validación. Si esa variación es del tamaño de la ventaja sobre 0.50, la ventaja no es confiable.

In [ ]:
filas = []
for s in (0, 1, 2):
    r = entrenar_mlp(
        X_all[idx_tr],
        y_all[idx_tr],
        X_all[idx_es],
        y_all[idx_es],
        PARAMS_FINAL,
        semilla=s,
    )
    mejor = r["historia"].loc[r["historia"]["epoca"] == r["epoca_mejor"]].iloc[0]
    filas.append(
        {"semilla": s, "epoca_mejor": r["epoca_mejor"], "auc_val": mejor["auc_val"]}
    )
semillas = pd.DataFrame(filas)
print(semillas.round(4).to_string(index=False))
print(
    f"AUC promedio: {semillas['auc_val'].mean():.4f} +- {semillas['auc_val'].std():.4f}"
)

## 8. Validación cruzada temporal del MLP (comparable con el notebook 02)

En cada uno de los 5 pliegues: se toma el entrenamiento del pliegue, se separan sus últimas fechas para el *early stopping*, se entrena la red y se evalúa en la ventana de validación del pliegue, que la red nunca vio. Se reporta AUC y F1 (con corte de 0.5, igual que las otras tablas).

In [ ]:
def cv_mlp(params, semilla=SEMILLA):
    filas = []
    for k, (tr, va) in enumerate(CV, 1):
        i_tr, i_es = dividir_por_fecha(fechas_all[tr], FRAC_ES, GAP)
        tr_in, es = tr[i_tr], tr[i_es]
        r = entrenar_mlp(
            X_all[tr_in], y_all[tr_in], X_all[es], y_all[es], params, semilla
        )
        p = proba(r, X_all[va])
        filas.append(
            {
                "pliegue": k,
                "epoca_mejor": r["epoca_mejor"],
                "auc": roc_auc_score(y_all[va], p),
                "f1": f1_score(y_all[va], (p >= 0.5).astype(int), zero_division=0),
            }
        )
    return pd.DataFrame(filas)


t0 = time.perf_counter()
cv_red = cv_mlp(PARAMS_FINAL)
print(f"Tiempo: {time.perf_counter() - t0:.0f} s")
cv_red.round(4)

In [ ]:
comparacion = pd.DataFrame(
    filas_ref
    + [
        {
            "modelo": "mlp_pytorch",
            "auc_cv": cv_red["auc"].mean(),
            "auc_std": cv_red["auc"].std(ddof=0),
            "f1_cv": cv_red["f1"].mean(),
        }
    ]
)
comparacion.round(4)

In [ ]:
orden = comparacion.sort_values("auc_cv")
fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.barh(orden["modelo"], orden["auc_cv"], xerr=orden["auc_std"], color="tab:blue")
ax.axvline(0.5, color="k", linestyle="--", linewidth=1, label="Sin información (0.50)")
ax.set_xlim(0.46, 0.56)
ax.set_title("AUC-ROC en validación cruzada temporal: red contra referencias")
ax.set_xlabel("AUC-ROC (promedio de pliegues)")
ax.set_ylabel("Modelo")
ax.legend()
plt.show()

**Cómo leer la comparación.** Si el AUC de la red queda dentro de una desviación del de la regresión logística (y de 0.50), no hay evidencia de que sirva más. Es un resultado válido y esperable. Si el F1 de la red queda por debajo del baseline, recuerda la razón del notebook 02: con 52.7 % de días al alza, decir siempre "sube" ya da un F1 alto en esa clase.

## 9. Modelo final

Se entrena la red con la combinación elegida sobre **todo el periodo de entrenamiento** (dejando las últimas fechas para el *early stopping*) y se queda listo para guardarse. El *holdout* de los últimos 6 meses sigue sin tocarse.

In [ ]:
res_final = entrenar_mlp(
    X_all[idx_tr],
    y_all[idx_tr],
    X_all[idx_es],
    y_all[idx_es],
    PARAMS_FINAL,
    verbose=True,
)
print("Época elegida:", res_final["epoca_mejor"])

## 10. Guardar con `torch.save()` y cargar sin reentrenar

Se guarda **un solo archivo** con todo lo necesario para predecir: los pesos (`state_dict`), la arquitectura, las columnas en orden y la media y desviación del escalador. Sin el escalador, la red recibiría números en otra escala que con la que aprendió y daría predicciones sin sentido. El archivo va a `models/` (Git lo ignora; lo versionará DVC).

Después se **carga en una instancia nueva** y se comprueba que da exactamente las mismas predicciones, como pide el documento.

In [ ]:
RUTA_MODELO = RAIZ / "models" / "mlp_exploracion.pt"
RUTA_MODELO.parent.mkdir(exist_ok=True)

esc = res_final["escalador"]
paquete = {
    "state_dict": res_final["modelo"].state_dict(),
    "arquitectura": {
        "n_entrada": len(COLS),
        "capas_ocultas": PARAMS_FINAL["capas_ocultas"],
        "dropout": PARAMS_FINAL["dropout"],
    },
    "columnas": COLS,
    "media": torch.as_tensor(esc.mean_, dtype=torch.float32),
    "escala": torch.as_tensor(esc.scale_, dtype=torch.float32),
    "epoca_mejor": res_final["epoca_mejor"],
    "auc_validacion": res_final["checkpoint"]["mejor_auc"],
}
torch.save(paquete, RUTA_MODELO)
print("Guardado:", RUTA_MODELO.name, f"({RUTA_MODELO.stat().st_size / 1024:.1f} KB)")

In [ ]:
def cargar_mlp(ruta):
    p = torch.load(ruta, weights_only=True)
    modelo = MLP(**p["arquitectura"])
    modelo.load_state_dict(p["state_dict"])
    modelo.eval()
    return modelo, p


def predecir_desde_archivo(ruta, X_crudo):
    modelo, p = cargar_mlp(ruta)
    x = (torch.as_tensor(X_crudo, dtype=torch.float32) - p["media"]) / p["escala"]
    with torch.no_grad():
        return torch.sigmoid(modelo(x)).numpy()


muestra = X_all[-500:]
p_memoria = proba(res_final, muestra)
p_archivo = predecir_desde_archivo(RUTA_MODELO, muestra)
assert np.allclose(p_memoria, p_archivo, atol=1e-5)
print("OK: el modelo cargado desde el archivo predice igual que el de memoria")
print("Primeras probabilidades:", np.round(p_archivo[:5], 4))

## 11. Holdout (una sola vez, al final)

Igual que en el notebook 02, esta celda está **apagada**. Se enciende solo cuando modelo, características e hiperparámetros ya no se vayan a cambiar.

In [ ]:
EVALUAR_HOLDOUT = False

if EVALUAR_HOLDOUT:
    p_hold = predecir_desde_archivo(RUTA_MODELO, hold[COLS].to_numpy())
    print("AUC holdout:", round(roc_auc_score(hold["y"], p_hold), 4))

## 12. Conclusiones

- **AUC del MLP en validación cruzada:** 0.5085 ± 0.014, contra 0.5093 de la regresión logística y 0.500 del baseline. El F1 es 0.667 (logística 0.678 y dummy 0.692).
- **¿La red aporta algo sobre la logística?** No. La diferencia (0.0008) es mucho menor que la desviación entre pliegues (0.014), y entre semillas el AUC de validación varía 0.501 ± 0.006. Con estas 13 características la señal es tan pequeña que una red no la aprovecha mejor que una recta.
- **Arquitectura elegida y época de parada:** dos capas ocultas de 32 y 16 neuronas, dropout 0.5 y sin `weight_decay`, elegida entre 12 combinaciones por AUC de validación (diferencias de milésimas, así que sirve sobre todo para documentar lo que se probó). El modelo final se queda con la época 2.
- **¿Early stopping o `epocas_max`?** En todas las corridas revisadas paró por *early stopping* (paciencia 10) antes de llegar a las 50 épocas. La época óptima cambia mucho entre pliegues (1 a 24) y entre semillas (1 a 16), otra señal de ruido.
- **Curvas:** la pérdida de entrenamiento baja un poco (0.692 a 0.690) y la de validación no baja de 0.693 (`ln 2`): la red no encuentra patrones que generalicen.
- **Serialización:** `torch.save` guarda pesos, arquitectura, columnas y escalador (7.9 KB); `torch.load(..., weights_only=True)` lo recarga y predice igual que el modelo en memoria, sin reentrenar.
- **Dos modelos finalistas del proyecto:** regresión logística (pipeline de scikit-learn) y este MLP (PyTorch). Su empate es un resultado en sí mismo y se reporta tal cual.

**Qué se mueve a `src/` después:** `DatasetAcciones` y `MLP` (módulo `models`), `entrenar_mlp` como método `train()` de una clase que herede de `BaseModel`, y `predecir_desde_archivo` como `predict()` con carga del archivo guardado, todo leyendo `configs/config.yaml`.